In [ ]:
import os
import sys
import subprocess
import numpy as np
from collections import Counter
from IPython.display import Image, display

lib_dir = os.getcwd()
if "notebooks" in lib_dir:
    lib_dir = os.path.join(lib_dir[:-10], 'PROFET')
os.chdir(lib_dir)
print("lib_dir:", lib_dir)

main_dir = os.path.dirname(lib_dir)
sys.path.insert(0, main_dir)  # for util.utils
sys.path.insert(0, lib_dir)   # for models.velocityfield

os.makedirs(os.path.join(main_dir, 'data'), exist_ok=True)
os.makedirs(os.path.join(main_dir, 'assets'), exist_ok=True)

from util.utils import ResourceMonitor, load_preprocessed_data, generate_animation, generate_W2distance_plot

contrast_colors = {
    2: '#1f77b4',   # blue
    4: '#ff7f0e',   # orange
    6: '#d62728',   # red
}

## LARRY_3000_benchmark dataset

### Data property

* 3 snapshots at day 2, 4, 6
* 3,000 genes · 49,302 total cells
* Training timepoints: `times=[2, 6]`, `d_red=2`
* Cells per training tp: Day 2: 4,638 · Day 6: 29,679
* Held-out / Intermediate: Day 4: 14,985

### Data source
Place `LARRY_3000_benchmark_preprocessed.pkl` in `data/`. The file is available from the shared benchmarks data directory.

In [ ]:
# Load preprocessed data
example_name = 'LARRY_3000_benchmark'

try:
    time_label, full_matrix, projected_matrix, pca = load_preprocessed_data(example_name)
except:
    time_label, full_matrix = load_preprocessed_data(example_name)

time_points = sorted(list(set(time_label)))
cnt = Counter(time_label)
print('Time points:', time_points)
print('Cell counts per time point:', dict(sorted(cnt.items())))
print('full_matrix shape:', full_matrix.shape)
print('projected_matrix shape:', projected_matrix.shape)

## PROFET

In [3]:
# -------------------------------------------------------
# User parameters
# -------------------------------------------------------
d_red             = 2
times             = [2, 6]       # training snapshots
intermediate_times = [4]         # held-out intermediate snapshot
dimension_reduction = 'Y'
exp_no            = 'benchmark'
exp_memo          = f'dim{d_red}_{exp_no}'

# GPA tunable parameters
adjust_lr_P_epochs    = 'Y'      # 'Y' to auto-adjust lr_P and epochs w.r.t. dimensionality
specify_f_Lip_threshold = 'Y'
f_Lip_threshold       = 1e-4

# ForceMatching tunable parameters
fm_iterations = 20000
fm_hidden_units = '64 64 64 64'

In [ ]:
# -------------------------------------------------------
# PROFET Step 1: GPA  (one pair: day 2 -> day 6)
# -------------------------------------------------------
os.chdir(lib_dir)

dimension_note = f'dim{d_red}-' if dimension_reduction in ['Y', 'yes'] else ''

gpa_filenames = []
with ResourceMonitor() as monitor:
    for i in range(len(times) - 1):
        cmd = (
            f'python3 run_GPA.py'
            f' --dataset {example_name}'
            f' --label {times[i]} {times[i+1]}'
            f' --N_dim {d_red}'
            f' --exp_no {exp_no}'
            f' --dimension_reduction {dimension_reduction}'
        )
        if adjust_lr_P_epochs in ['N', 'no']:
            cmd += f' --adjust_lr_P_epochs {adjust_lr_P_epochs}'
        if specify_f_Lip_threshold in ['Y', 'yes']:
            cmd += f' --f_Lip_threshold {f_Lip_threshold}'

        print(f'Running GPA: day {times[i]} -> day {times[i+1]}')
        subprocess.run(cmd, shell=True, check=True)

        N_Q = int(cnt[times[i+1]])
        N_P = int(cnt[times[i]])
        fname = f'KL-Lipschitz_1.0000-{times[i]}_{times[i+1]}times-{dimension_note}{N_Q:04d}_{N_P:04d}-00-{exp_no}.pickle'
        gpa_filenames.append(fname)

result_dir = os.path.join(main_dir, 'assets', example_name, exp_memo)
os.makedirs(result_dir, exist_ok=True)
monitor.report('PROFET GPA', dataset=example_name, d_red=d_red,
               filename=os.path.join(result_dir, 'resources.txt'))

print('GPA output files:', gpa_filenames)

In [ ]:
# -------------------------------------------------------
# PROFET Step 2: Force Matching
# -------------------------------------------------------
os.chdir(lib_dir)

times_join         = ' '.join(str(t) for t in times)
gpa_filenames_join = ' '.join(gpa_filenames)

cmd = (
    f'python3 run_ForceMatching.py'
    f' --dataset {example_name}'
    f' --ts {times_join}'
    f' --exp_memo {exp_memo}'
    f' --files {gpa_filenames_join}'
    f' --iterations {fm_iterations}'
    f' --hidden_units {fm_hidden_units}'
)

with ResourceMonitor() as monitor:
    print('Running Force Matching...')
    subprocess.run(cmd, shell=True, check=True)

monitor.report('PROFET ForceMatching', dataset=example_name, d_red=d_red,
               filename=os.path.join(result_dir, 'resources.txt'))

In [ ]:
# -------------------------------------------------------
# Load velocity field and integrate ODE
# -------------------------------------------------------
from models.velocityfield import VelocityField

velocity_net, p = VelocityField.load(os.path.join(main_dir, 'assets', example_name, exp_memo, ''))

dt          = p['numerical_ts'][-1] / 200
numerical_dt = dt
physical_dt  = dt * p['ts'][-1] / p['numerical_ts'][-1]

input_data = projected_matrix[time_label == times[0], :d_red]

X1_trpts = velocity_net.integrate(input_data, T=p['numerical_ts'][-1], dt=dt)
print(f'Trajectory frames: {len(X1_trpts)}, shape per frame: {X1_trpts[0].shape}')

# Save trajectories
traj_npy = os.path.join(result_dir, 'backward_trajectories.npy')
np.save(traj_npy, np.stack(X1_trpts, axis=0))
print(f'Saved trajectory to {traj_npy}')

In [ ]:
img_src1 = os.path.join(result_dir, 'movie-particles.gif')
img_src2 = os.path.join(result_dir, 'movie-velocities.gif')
img_src3 = os.path.join(result_dir, 'w2distances.png')

generate_animation(example_name, times, intermediate_times, X1_trpts,
                   img_src1, d_red, True, colors=contrast_colors, plot_vectorfield=False)
display(Image(filename=img_src1))

generate_animation(example_name, times, intermediate_times, X1_trpts,
                   img_src2, d_red, True, colors=contrast_colors, plot_vectorfield=True)
display(Image(filename=img_src2))

generate_W2distance_plot(example_name, times, intermediate_times, X1_trpts,
                         img_src3, d_red, True, colors=contrast_colors)
display(Image(filename=img_src3))